In [3]:
#import the pandas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

#Read the datasets
customers = pd.read_csv('customers.csv')
edges= pd.read_csv('network_edges.csv')
Outcomes = pd.read_csv('campaign_outcomes.csv')

df = customers
df.head()



,customer_id,age,gender,income_band,region,preferred_channel,baseline_propensity,segment
0,C00001,22,F,mid,West,store,0.211116,new
1,C00002,58,M,mid,East,store,0.249371,loyal
2,C00003,52,M,low,South,store,0.054098,loyal
3,C00004,40,M,mid,North,omnichannel,0.136738,value
4,C00005,40,F,mid,North,online,0.189661,value


# Data Cleaning Steps

In [5]:
#Basic data checks
df.info()
df.shape
df.describe()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   customer_id          5000 non-null   object 
 1   age                  5000 non-null   int64  
 2   gender               5000 non-null   object 
 3   income_band          5000 non-null   object 
 4   region               5000 non-null   object 
 5   preferred_channel    5000 non-null   object 
 6   baseline_propensity  5000 non-null   float64
 7   segment              5000 non-null   object 
dtypes: float64(1), int64(1), object(6)
memory usage: 312.6+ KB


,age,baseline_propensity
count,5000.000000,5000.000000
mean,43.424400,0.167324
std,15.047577,0.067552
min,18.000000,0.050017
25%,30.000000,0.116016
50%,43.000000,0.158542
75%,57.000000,0.215238
max,69.000000,0.348934


In [6]:
#Checking the missing data if any

df.isna().count()

#There is no missing data values

,0
customer_id,5000
age,5000
gender,5000
income_band,5000
region,5000
preferred_channel,5000
baseline_propensity,5000
segment,5000


In [7]:
#check duplicates
df.drop_duplicates()

#There is no dupliactes in the tables.

,customer_id,age,gender,income_band,region,preferred_channel,baseline_propensity,segment
0,C00001,22,F,mid,West,store,0.211116,new
1,C00002,58,M,mid,East,store,0.249371,loyal
2,C00003,52,M,low,South,store,0.054098,loyal
3,C00004,40,M,mid,North,omnichannel,0.136738,value
4,C00005,40,F,mid,North,online,0.189661,value
...,...,...,...,...,...,...,...,...
4995,C04996,29,M,high,North,omnichannel,0.151950,new
4996,C04997,56,F,mid,West,online,0.159469,loyal
4997,C04998,60,M,mid,South,omnichannel,0.183228,loyal
4998,C04999,53,M,low,East,online,0.066058,loyal


In [8]:
#Print all the datasets' shapes

print("customers",df.shape)
print("Edges", edges.shape)
print("Outcomes", Outcomes.shape)

customers (5000, 8)
Edges (40000, 4)
Outcomes (5000, 11)


In [10]:
#Merge outcome with customer attributes

outcomes_with_attrs = Outcomes.merge(customers, on='customer_id')
outcomes_with_attrs.head()

,customer_id,treatment_group,num_treated_neighbors,share_treated_neighbors,converted,orders,revenue,used_direct_coupon,used_referral_code,profit,exposure_bucket,age,gender,income_band,region,preferred_channel,baseline_propensity,segment
0,C00001,1,4,0.500000,1,3,36.66,1,0,14.66,both,22,F,mid,West,store,0.211116,new
1,C00002,1,5,0.625000,1,1,21.40,1,0,8.56,both,58,M,mid,East,store,0.249371,loyal
2,C00003,0,6,0.750000,0,0,0.00,0,0,0.00,spillover_only,52,M,low,South,store,0.054098,loyal
3,C00004,0,6,0.666667,1,1,13.57,0,1,5.43,spillover_only,40,M,mid,North,omnichannel,0.136738,value
4,C00005,0,6,0.750000,0,0,0.00,0,0,0.00,spillover_only,40,F,mid,North,online,0.189661,value


In [14]:
#treatment balance
print(outcomes_with_attrs.groupby('treatment_group').size())
print('\nConversion and revenue by treatment (naive glimpse)')
print(outcomes_with_attrs.groupby('treatment_group').agg(
    converted = ('converted', 'mean'),
    revenue = ('revenue', 'mean'),
    profit =('profit','mean')
)
)

treatment_group
0    2500
1    2500
dtype: int64

Conversion and revenue by treatment (naive glimpse)
                 converted    revenue     profit
treatment_group                                 
0                   0.3016  12.253724   5.505832
1                   0.5260  27.972364  11.943632


In [16]:
#Exposure buckets

print('Exposure buckets (network-aware view):')
print(outcomes_with_attrs['exposure_bucket'].value_counts().sort_index())

Exposure buckets (network-aware view):
exposure_bucket
both              2499
direct_only         90
pure_control        10
spillover_only    2401
Name: count, dtype: int64


In [19]:
#By Region and Segment

print("by Region")
print(outcomes_with_attrs.groupby(['region', 'treatment_group']).agg(
    n=('customer_id','count'),
    cr=('converted','mean'), #conversion rate
    arpc=('revenue','mean') #avg_revenue_per_customer
).round(4)
)

by Region
                          n      cr     arpc
region treatment_group                      
East   0                611  0.3093  12.1309
       1                632  0.5316  28.5447
North  0                630  0.3127  13.6217
       1                613  0.5106  25.8541
South  0                613  0.3018  11.4966
       1                623  0.5201  28.0030
West   0                646  0.2833  11.7542
       1                632  0.5411  29.4244
